In [ ]:
import os
import warnings
import roman_datamodels as rdm
import pandas as pd

from tqdm import tqdm
from astropy.time import Time
from romancal.assign_wcs import AssignWcsStep
import logging

# silence only the AssignWcsStep logger
# wcs_logger = logging.getLogger("stpipe.AssignWcsStep")
# wcs_logger.setLevel(logging.CRITICAL)  # Only CRITICAL errors will print
logging.getLogger("stpipe").setLevel(logging.CRITICAL)
logging.disable(logging.INFO)

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(
    "/Users/jimartin/Work/ROMAN/TRExS/simulations/soc_bgtds/roman_soc_sim_gbtds_tile1_wfi01_f146.csv",
    names=["s3_path"],
)
df

,s3_path
0,stpubdata/roman/nexus/soc_simulations/r00340/l...
1,stpubdata/roman/nexus/soc_simulations/r00340/l...
2,stpubdata/roman/nexus/soc_simulations/r00340/l...
3,stpubdata/roman/nexus/soc_simulations/r00340/l...
4,stpubdata/roman/nexus/soc_simulations/r00340/l...
...,...
738,stpubdata/roman/nexus/soc_simulations/r00340/l...
739,stpubdata/roman/nexus/soc_simulations/r00340/l...
740,stpubdata/roman/nexus/soc_simulations/r00340/l...
741,stpubdata/roman/nexus/soc_simulations/r00340/l...


In [3]:
for k, v in os.environ.items():
    if "CRDS" in k:
        print(f"{k}: {v}")

CRDS_PATH: /Users/jimartin/data/crds_cache/
CRDS_SERVER_URL: https://roman-crds.stsci.edu
CRDS_FITS_IGNORE_MISSING_END: false
CRDS_FITS_VERIFY_CHECKSUM: true
CRDS_ADD_LOG_MSG_COUNTER: false
CRDS_ALLOW_BAD_REFERENCES: false
CRDS_ALLOW_BAD_RULES: false
CRDS_ALLOW_SCHEMA_VIOLATIONS: false
CRDS_ALLOW_BAD_PARKEY_VALUES: false
CRDS_ALLOW_BAD_USEAFTER: false
CRDS_USE_PICKLED_CONTEXTS: false
CRDS_AUTO_PICKLE_CONTEXTS: false
CRDS_FORCE_COMPLETE_LOAD: false
CRDS_EXPLICIT_GARBAGE_COLLECTION: true
CRDS_MODE: auto
CRDS_IGNORE_MAPPING_CHECKSUM: false
CRDS_DOWNLOAD_MODE: http
CRDS_CONFIG_URI: None
CRDS_MAPPING_URI: None
CRDS_REFERENCE_URI: None
CRDS_PICKLE_URI: None
CRDS_DOWNLOAD_CHECKSUMS: true
CRDS_DOWNLOAD_LENGTHS: true
CRDS_CLIENT_RETRY_COUNT: 1
CRDS_CLIENT_RETRY_DELAY_SECONDS: 0
CRDS_CLIENT_TIMEOUT_SECONDS: 3600
CRDS_LOCK_PATH: /tmp
CRDS_USE_LOCKING: true
CRDS_LOCKING_MODE: multiprocessing
CRDS_S3_ENABLED: false
CRDS_S3_RETURN_URI: false
CRDS_OBSERVATORY: None
CRDS_S3_BUCKET: None


In [28]:
for f in tqdm(df["s3_path"].values[600:]):
    f = os.path.basename(f)
    path = f"/Users/jimartin/Work/ROMAN/TRExS/simulations/soc_bgtds/{f}"
    # print(f"Processing {path}...")
    dm = rdm.open(path, lazy_load=False)
    org_date = dm.meta.file_date

    # fix gwcs
    dm = AssignWcsStep.call(dm)

    # WORKAROUND: Force file_date to a standard string
    # so the to_asdf context manager doesn't crash on DNode casting
    if hasattr(dm.meta, "file_date"):
        dm.meta.file_date = Time.now()

    dm.to_asdf(path.replace(".asdf", "_gwcs.asdf"))

    # break

100%|██████████| 143/143 [02:53<00:00,  1.21s/it]
